# Spinal Disorder Classification — Stage 3F: Multi-Seed Stability

**Goal:** determine whether the promising CNN validation results persist when we retrain from independent random initializations. This notebook compares the **same reference CNN** with three preprocessing methods:

| Candidate | Input preprocessing | Historical result (seed 42) |
|---|---|---|
| **E0-G1** | Grayscale, 128×128 | 49/52 = 94.23%, *minimum-loss* checkpoint |
| **C01** | Grayscale + CLAHE clip 1.0, 4×4 tiles | 49/52 = 94.23%, *minimum-loss* checkpoint |
| **C08** | Grayscale + CLAHE clip 3.0, 8×8 tiles | **50/52 = 96.15%**, *maximum-accuracy* checkpoint in the C08-R1 reproduction |

**New independent model seeds:** `123`, `456`, `777`, tested for **each candidate**: 9 training runs total. The frozen **dataset split seed stays 42**; we do not resplit. For each new run we save both `best_loss_model.keras` and `best_accuracy_model.keras` and recompute metrics from the saved files. **Do not mix checkpoint policies** when ranking candidates.

**Predeclared primary comparison:** mean **validation macro F1 of the minimum-validation-loss checkpoint** over the three new seeds; inspect balanced accuracy, worst-seed performance and loss too. Best-accuracy checkpoints are **exploratory** and included to assess whether the 96.15% peak is robust. The historical seed-42 results provide context but **are not pooled** with the new nine runs because their checkpoint policies differ.

**Research limitations:** All runs share just **52 validation X-rays**. This measures sensitivity to initialization, **not** independent validation or patient-independent generalization. Repeated selection on these images risks overfitting the validation set. The 51 held-out test X-rays remain untouched; the images and filenames stay private. Not a clinical diagnostic device.

**How to run:** Upload to Colab, preferably choose a GPU runtime, then Runtime → Run all. On CPU, nine trainings may take roughly 40–90 minutes. You can run one seed at a time by editing `RUN_SEEDS` in Section 2; completed matching runs are skipped and remain saved to Google Drive.


## 1. Mount your original Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Imports, fixed dataset, candidates and **new** seed choices

In [ ]:
from pathlib import Path
import hashlib
import json
import random
import time
import platform

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from IPython.display import display
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score,
)
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf

SPLIT_SEED = 42  # This verifies the existing dataset split, NOT the new model seeds.
DATA_ROOT = Path('/content/drive/MyDrive/224')
SPLIT_DIR = Path('/content/drive/MyDrive/Spinal_Classification_Optimization/splits/v1')
EXP_DIR = Path('/content/drive/MyDrive/Spinal_Classification_Optimization/experiments/stage3')
STABILITY_DIR = EXP_DIR / 'stability_v1'
STABILITY_DIR.mkdir(parents=True, exist_ok=True)

LABEL_TO_NAME = {0:'Normal', 1:'Scoliosis', 2:'Spondylolisthesis'}
CLASS_TO_LABEL = {'Normal':0, 'Scol':1, 'Spond':2}

CANDIDATES = {
    'E0-G1': {'mode':'gray', 'clip_limit':None, 'tile_grid':None},
    'C01': {'mode':'clahe', 'clip_limit':1.0, 'tile_grid':(4, 4)},
    'C08': {'mode':'clahe', 'clip_limit':3.0, 'tile_grid':(8, 8)},
}
NEW_SEEDS = (123, 456, 777)
RUN_MODELS = ['E0-G1', 'C01', 'C08']
RUN_SEEDS = [123, 456, 777]  # To run incrementally: use [123], then [456], then [777].
EPOCHS = 40
BATCH_SIZE = 16

assert set(RUN_MODELS) <= set(CANDIDATES)
assert len(set(RUN_MODELS)) == len(RUN_MODELS)
assert set(RUN_SEEDS) <= set(NEW_SEEDS) and len(set(RUN_SEEDS)) == len(RUN_SEEDS)

print('TensorFlow:', tf.__version__, '| OpenCV:', cv2.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))
print('New results folder:', STABILITY_DIR)
print('Planned models:', RUN_MODELS, '| seeds:', RUN_SEEDS)
if not tf.config.list_physical_devices('GPU'):
    print('CPU-only runtime: consider a Colab GPU and/or running one seed at a time.')


## 3. Verify frozen Stage 2 split and dataset fingerprint (no new split)

In [ ]:
manifest_path = SPLIT_DIR / 'split_manifest_PRIVATE.csv'
config_path = SPLIT_DIR / 'split_config_PRIVATE.json'

assert manifest_path.is_file() and config_path.is_file(), (
    'Frozen Stage 2 split not found. Run Stage 2 first; do not generate a new split.'
)
manifest = pd.read_csv(manifest_path, keep_default_na=False)
freeze = json.loads(config_path.read_text(encoding='utf-8'))

required = {'relative_path', 'class_name', 'label', 'file_sha256', 'group_id', 'split'}
assert required <= set(manifest.columns), f'Missing manifest columns: {required - set(manifest.columns)}'
assert len(manifest) == 338 and manifest['relative_path'].is_unique
assert freeze['class_to_label'] == CLASS_TO_LABEL
assert freeze['seed'] == SPLIT_SEED
assert freeze['patient_independent'] is False
assert set(manifest['split']) == {'train', 'validation', 'test'}
assert manifest.groupby('group_id')['split'].nunique().max() == 1, 'Grouped images cross splits.'
assert all(int(row.label) == CLASS_TO_LABEL[row.class_name] for row in manifest.itertuples(index=False))

rows = (manifest[['relative_path', 'class_name', 'label', 'file_sha256']]
        .sort_values('relative_path')
        .to_dict(orient='records'))
fingerprint = hashlib.sha256(
    json.dumps(rows, sort_keys=True, separators=(',', ':')).encode('utf-8')
).hexdigest()
assert fingerprint == freeze['dataset_fingerprint'], 'Frozen dataset fingerprint mismatch.'

counts = pd.crosstab(manifest['class_name'], manifest['split']).reindex(
    index=list(CLASS_TO_LABEL), columns=['train', 'validation', 'test'], fill_value=0
)
display(counts.assign(total=counts.sum(axis=1)))
assert counts['train'].sum() == 235 and counts['validation'].sum() == 52 and counts['test'].sum() == 51

train_df = manifest[manifest['split'] == 'train'].sort_values('relative_path').reset_index(drop=True)
val_df = manifest[manifest['split'] == 'validation'].sort_values('relative_path').reset_index(drop=True)
# Intentional: no test_df or test image loader in this notebook.
assert set(train_df['relative_path']).isdisjoint(val_df['relative_path'])
print('Freeze verified. Development images:', len(train_df) + len(val_df))
print('Test set RESERVEd for final evaluation:', int(counts['test'].sum()))

## 4. Read the original reference settings and verify candidate provenance

We verify the saved seed-42 model configurations so we do not inadvertently compare a different architecture or preprocessing method.

In [ ]:
HISTORICAL = {
    'E0-G1': EXP_DIR / 'E0-G1',
    'C01': EXP_DIR / 'clahe_grid_v1' / 'C01',
    'C08': EXP_DIR / 'clahe_grid_v1' / 'C08-R1',
}

original_configs = {}
for candidate, path in HISTORICAL.items():
    conf_path = path / 'configuration.json'
    assert conf_path.is_file(), f'Missing previous {candidate} configuration: {conf_path}'
    cfg = json.loads(conf_path.read_text(encoding='utf-8'))
    original_configs[candidate] = cfg
    assert cfg['dataset_fingerprint'] == fingerprint
    assert cfg['split_version'] == 'v1' and cfg['seed'] == SPLIT_SEED
    assert cfg.get('architecture') == 'reference'
    assert cfg.get('batch_size') == BATCH_SIZE and cfg.get('epochs_max') == EPOCHS
    assert cfg.get('test_evaluated', cfg.get('test_set_evaluated')) is False
    assert cfg.get('parameters') == 4287491, 'Unexpected saved reference architecture size.'
    if candidate == 'E0-G1':
        assert cfg.get('mode') == 'gray' and cfg.get('size') == 128
        assert cfg.get('initial_learning_rate') == 3e-4
    else:
        assert cfg.get('clip_limit') == CANDIDATES[candidate]['clip_limit']
        assert tuple(cfg.get('tile_grid')) == CANDIDATES[candidate]['tile_grid']
        assert cfg.get('learning_rate') == 3e-4

assert (HISTORICAL['C08'] / 'best_accuracy_model.keras').is_file(), 'C08-R1 best accuracy checkpoint missing.'
print('Historical reference settings and C08-R1 checkpoint verified.')


## 5. Deterministic preprocessing (same as Stages 3B–3E)

All images use the same EXIF orientation, RGB conversion, LANCZOS padding to 128×128, grayscale conversion, optional CLAHE, and division by 255. Data are loaded only from **train** and **validation** partitions. The arrays are cached per candidate within this runtime to avoid rereading them nine times.

In [ ]:
def file_sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def preprocess_image(path, setting):
    with Image.open(path) as opened:
        original = ImageOps.exif_transpose(opened).convert('RGB')
        resized = ImageOps.pad(
            original, (128, 128), method=Image.Resampling.LANCZOS,
            color=(0, 0, 0)
        )
        rgb = np.asarray(resized, dtype=np.uint8)

    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    if setting['mode'] == 'clahe':
        gray = cv2.createCLAHE(
            clipLimit=float(setting['clip_limit']),
            tileGridSize=tuple(setting['tile_grid'])
        ).apply(gray)
    elif setting['mode'] != 'gray':
        raise ValueError('Unknown preprocessing mode')

    return gray[..., None].astype(np.float32) / 255.0


def load_partition(rows, setting):
    X, y = [], []
    for record in rows.itertuples(index=False):
        path = DATA_ROOT / record.relative_path
        if not path.is_file() or file_sha256(path) != record.file_sha256:
            raise RuntimeError(f'Missing/changed image in frozen manifest: {record.relative_path}')
        X.append(preprocess_image(path, setting))
        y.append(int(record.label))
    return np.stack(X), np.asarray(y, dtype=np.int64)


DATA_CACHE = {}
def get_candidate_data(candidate):
    if candidate not in DATA_CACHE:
        setting = CANDIDATES[candidate]
        X_train, y_train = load_partition(train_df, setting)
        X_val, y_val = load_partition(val_df, setting)
        assert X_train.shape == (235,128,128,1) and X_val.shape == (52,128,128,1)
        assert X_train.dtype == np.float32 and np.isfinite(X_train).all()
        assert 0 <= X_train.min() <= X_train.max() <= 1
        assert 0 <= X_val.min() <= X_val.max() <= 1
        DATA_CACHE[candidate] = (X_train, y_train, X_val, y_val)
        print('Loaded', candidate, ': 235 training and 52 validation images (no test images).')
    return DATA_CACHE[candidate]

# Verify that grayscale and CLAHE produce different arrays on a training image.
sample_file = DATA_ROOT / train_df.iloc[0]['relative_path']
sample_gray = preprocess_image(sample_file, CANDIDATES['E0-G1'])
sample_clahe = preprocess_image(sample_file, CANDIDATES['C08'])
assert sample_gray.shape == sample_clahe.shape == (128,128,1)
assert not np.array_equal(sample_gray, sample_clahe), 'CLAHE did not modify the sample.'
print('Preprocessing sanity checks passed.')


## 6. Same reference CNN, loss function, optimizer and class weights

Training runs differ **only** in preprocessing and random seed. Mild augmentation, BatchNormalization, dropout, and other architectural changes are not introduced.

In [ ]:
from tensorflow.keras import layers


def make_reference_cnn(shape):
    return tf.keras.Sequential([
        layers.Input(shape=shape),
        layers.Conv2D(32, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(),
        layers.Conv2D(128, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dense(3, activation='softmax'),
    ], name='reference_cnn')


def make_model(shape):
    model = make_reference_cnn(shape)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=3e-4),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=['accuracy']
    )
    return model

preview = make_model((128,128,1))
assert preview.count_params() == 4287491
print('Verified exact reference CNN parameter count:', preview.count_params())
del preview
tf.keras.backend.clear_session()


## 7. Preserve two distinct checkpoints: minimum loss and maximum accuracy

For best-accuracy checkpoint selection, validation loss breaks ties. This is an explicitly **exploratory** policy. Never describe a peak accuracy as if it were an independently evaluated model.

In [ ]:
class SaveBestAccuracy(tf.keras.callbacks.Callback):
    def __init__(self, filepath):
        super().__init__()
        self.filepath = str(filepath)
        self.best_accuracy = float('-inf')
        self.best_loss = float('inf')
        self.best_epoch = None

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        accuracy = float(logs['val_accuracy'])
        loss = float(logs['val_loss'])
        better_accuracy = accuracy > self.best_accuracy + 1e-8
        tied_lower_loss = (
            abs(accuracy - self.best_accuracy) <= 1e-8
            and loss < self.best_loss - 1e-8
        )
        if better_accuracy or tied_lower_loss:
            self.best_accuracy = accuracy
            self.best_loss = loss
            self.best_epoch = epoch + 1
            self.model.save(self.filepath)


def checkpoint_metrics(filepath, X_val, y_val):
    saved_model = tf.keras.models.load_model(str(filepath))
    scores = saved_model.predict(X_val, batch_size=BATCH_SIZE, verbose=0)
    pred = np.argmax(scores, axis=1)
    cm = confusion_matrix(y_val, pred, labels=[0,1,2])
    report = classification_report(
        y_val, pred, labels=[0,1,2],
        target_names=[LABEL_TO_NAME[i] for i in range(3)],
        output_dict=True, zero_division=0
    )
    return {
        'val_accuracy': float(accuracy_score(y_val, pred)),
        'correct': int((pred == y_val).sum()),
        'val_macro_f1': float(f1_score(
            y_val, pred, labels=[0,1,2], average='macro', zero_division=0
        )),
        'val_balanced_accuracy': float(balanced_accuracy_score(y_val, pred)),
        'per_class_recall': {
            name: float(report[name]['recall']) for name in LABEL_TO_NAME.values()
        },
        'confusion_matrix': cm.tolist()
    }


## 8. Train each model–seed pair (append-only; safe to resume)

Each pair has its own folder: `experiments/stage3/stability_v1/<candidate>/seed_<number>/`. If a complete, compatible result exists, it will be **skipped**. If an incomplete folder exists, the notebook stops and asks you to review it rather than deleting anything.

In [ ]:
def train_one(candidate, seed):
    if candidate not in CANDIDATES or seed not in NEW_SEEDS:
        raise ValueError('Not a preregistered candidate and seed')
    setting = CANDIDATES[candidate]
    folder = STABILITY_DIR / candidate / f'seed_{seed}'
    config_path = folder / 'configuration.json'
    results_path = folder / 'validation_metrics.json'
    best_loss_path = folder / 'best_loss_model.keras'
    best_accuracy_path = folder / 'best_accuracy_model.keras'
    history_path = folder / 'history.csv'

    expected = {
        'stage': '3F', 'candidate': candidate, 'model_seed': int(seed),
        'split_seed': SPLIT_SEED, 'dataset_fingerprint': fingerprint,
        'split_version': 'v1', 'architecture': 'reference',
        'input_mode': setting['mode'],
        'clip_limit': setting['clip_limit'],
        'tile_grid': list(setting['tile_grid']) if setting['tile_grid'] else None,
        'epochs_max': EPOCHS, 'batch_size': BATCH_SIZE,
        'learning_rate': 3e-4,
        'checkpoint_policy': 'best_val_loss_and_best_val_accuracy_loss_tiebreak',
    }
    if folder.exists():
        required = [config_path, results_path, best_loss_path, best_accuracy_path,
                    history_path, folder / 'epoch_log.csv', folder / 'learning_curves.png']
        if not all(p.is_file() for p in required):
            raise RuntimeError(
                f'Incomplete saved result at {folder}. Preserve it and inspect manually. '
                'Never auto-delete or overwrite past training artifacts.'
            )
        saved_config = json.loads(config_path.read_text(encoding='utf-8'))
        for key, val in expected.items():
            if saved_config.get(key) != val:
                raise RuntimeError(f'Existing {folder} differs on {key}; stop.')
        saved_metrics = json.loads(results_path.read_text(encoding='utf-8'))
        assert saved_metrics.get('test_evaluated') is False
        print(f'SKIP completed {candidate} seed {seed}.')
        return saved_metrics

    X_train, y_train, X_val, y_val = get_candidate_data(candidate)
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(int(seed))
    model = make_model(X_train.shape[1:])
    assert model.count_params() == 4287491
    weights = compute_class_weight('balanced', classes=np.arange(3), y=y_train)
    class_weights = {int(k):float(w) for k,w in enumerate(weights)}

    configuration = {
        **expected,
        'class_weights': class_weights,
        'augmentation': 'none',
        'normalization': 'divide grayscale / CLAHE pixels by 255',
        'resize': [128,128],
        'early_stopping': 'val_loss patience=9 restore_best_weights=True',
        'lr_schedule': 'val_loss factor=0.5 patience=4 min_lr=1e-6',
        'train_images': 235, 'validation_images': 52,
        'test_evaluated': False, 'patient_independent': False,
        'tensorflow_version': str(tf.__version__),
        'opencv_version': str(cv2.__version__),
        'python_version': platform.python_version(),
        'gpu_names': [d.name for d in tf.config.list_physical_devices('GPU')],
        'parameter_count': int(model.count_params()),
    }

    folder.mkdir(parents=True, exist_ok=False)
    config_path.write_text(json.dumps(configuration, indent=2), encoding='utf-8')
    acc_saver = SaveBestAccuracy(best_accuracy_path)
    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(
            str(best_loss_path), monitor='val_loss', save_best_only=True
        ),
        acc_saver,
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=9, restore_best_weights=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6
        ),
        tf.keras.callbacks.CSVLogger(str(folder / 'epoch_log.csv'))
    ]

    print(f'\n=== {candidate} | model seed {seed} | {setting} ===')
    started = time.perf_counter()
    training = model.fit(
        X_train, y_train, validation_data=(X_val, y_val),
        epochs=EPOCHS, batch_size=BATCH_SIZE,
        class_weight=class_weights, shuffle=True,
        callbacks=callbacks, verbose=2
    )
    pd.DataFrame(training.history).to_csv(history_path, index=False)

    best_loss = checkpoint_metrics(best_loss_path, X_val, y_val)
    best_acc = checkpoint_metrics(best_accuracy_path, X_val, y_val)
    loss_epoch = int(np.argmin(training.history['val_loss']) + 1)
    peak_epochs = (np.flatnonzero(np.isclose(
        training.history['val_accuracy'], max(training.history['val_accuracy']), atol=1e-6
    )) + 1).tolist()
    assert abs(best_acc['val_accuracy'] - acc_saver.best_accuracy) < 1e-5
    assert abs(best_loss['val_accuracy'] - training.history['val_accuracy'][loss_epoch - 1]) < 1e-5
    assert acc_saver.best_epoch in peak_epochs

    best_loss['selected_epoch'] = loss_epoch
    best_loss['val_loss'] = float(np.min(training.history['val_loss']))
    best_acc['selected_epoch'] = int(acc_saver.best_epoch)
    best_acc['val_loss'] = float(acc_saver.best_loss)

    metrics = {
        'candidate': candidate, 'model_seed': int(seed),
        'best_loss': best_loss, 'best_accuracy': best_acc,
        'epochs_completed': int(len(training.history['loss'])),
        'runtime_seconds': float(time.perf_counter() - started),
        'test_evaluated': False,
    }
    for policy, checkpoint in [('best_loss', best_loss), ('best_accuracy', best_acc)]:
        np.save(folder / f'confusion_matrix_{policy}_validation.npy',
                np.asarray(checkpoint['confusion_matrix'], dtype=np.int64))

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for metric, ax in [('loss', axes[0]), ('accuracy', axes[1])]:
        ax.plot(training.history[metric], label='train')
        ax.plot(training.history['val_' + metric], label='validation')
        ax.set_xlabel('Epoch')
        ax.set_title(f'{candidate} seed {seed}: {metric}')
        ax.legend()
    fig.tight_layout()
    fig.savefig(folder / 'learning_curves.png', dpi=140)
    plt.show()
    plt.close(fig)

    # The metrics marker is written only after every checkpoint and summary is saved.
    results_path.write_text(json.dumps(metrics, indent=2), encoding='utf-8')
    print(f'{candidate} seed {seed} | best-loss: '
          f'{best_loss["correct"]}/52, macro F1 {best_loss["val_macro_f1"]:.4f} | '
          f'best-accuracy: {best_acc["correct"]}/52, macro F1 {best_acc["val_macro_f1"]:.4f}')
    return metrics


# Each call saves its checkpoints to Drive; if Colab disconnects, rerun safely.
run_results = []
for candidate in RUN_MODELS:
    for model_seed in RUN_SEEDS:
        run_results.append(train_one(candidate, model_seed))

print('Requested training runs finished or reused:', len(run_results))


## 9. Compare seeds with **one consistent checkpoint policy**

The primary table uses **minimum validation loss**, which matches the original Stage 3 CNN checkpoint policy. The best-accuracy table is a **separate exploratory analysis**. Report mean, standard deviation (n=3), minimum and maximum, and the individual seeds. The three seeds share the same 52 validation images and therefore do not provide independent validation samples.

In [ ]:
def summarize_saved(policy):
    records = []
    for candidate in CANDIDATES:
        for seed in NEW_SEEDS:
            folder = STABILITY_DIR / candidate / f'seed_{seed}'
            config_path = folder / 'configuration.json'
            path = folder / 'validation_metrics.json'
            if not path.is_file():
                continue
            if not config_path.is_file():
                raise RuntimeError(f'Missing configuration for {folder}')
            cfg = json.loads(config_path.read_text(encoding='utf-8'))
            assert cfg.get('dataset_fingerprint') == fingerprint
            assert cfg.get('candidate') == candidate and cfg.get('model_seed') == seed
            result = json.loads(path.read_text(encoding='utf-8'))
            assert result.get('test_evaluated') is False
            ckpt = result[policy]
            records.append({
                'candidate': candidate, 'seed': seed, 'checkpoint': policy,
                'accuracy': ckpt['val_accuracy'], 'macro_f1': ckpt['val_macro_f1'],
                'balanced_accuracy': ckpt['val_balanced_accuracy'],
                'val_loss': ckpt['val_loss'], 'correct_of_52': ckpt['correct'],
                'epoch': ckpt['selected_epoch'],
                'recall_normal': ckpt['per_class_recall']['Normal'],
                'recall_scoliosis': ckpt['per_class_recall']['Scoliosis'],
                'recall_spondylolisthesis': ckpt['per_class_recall']['Spondylolisthesis'],
            })
    return pd.DataFrame.from_records(records)


def show_summary(policy):
    frame = summarize_saved(policy)
    print(f'\n--- {policy} ---')
    if frame.empty:
        print('No completed runs yet.')
        return frame
    display(frame.sort_values(['candidate', 'seed']).round(4))
    all_3 = []
    for candidate, group in frame.groupby('candidate'):
        if len(group) == len(NEW_SEEDS) and set(group['seed']) == set(NEW_SEEDS):
            all_3.append({
                'candidate': candidate,
                'accuracy_mean': group['accuracy'].mean(),
                'accuracy_std': group['accuracy'].std(ddof=1),
                'macro_f1_mean': group['macro_f1'].mean(),
                'macro_f1_std': group['macro_f1'].std(ddof=1),
                'macro_f1_min': group['macro_f1'].min(),
                'balanced_accuracy_mean': group['balanced_accuracy'].mean(),
                'val_loss_mean': group['val_loss'].mean(),
                'val_loss_std': group['val_loss'].std(ddof=1),
            })
        else:
            print(f'{candidate}: only {len(group)}/3 seeds completed. Not ranked yet.')
    if all_3:
        summary = pd.DataFrame(all_3).sort_values('macro_f1_mean', ascending=False)
        print('Complete three-seed candidates only:')
        display(summary.round(4))
    return frame

print('PRIMARY PROTOCOL: best validation LOSS checkpoints')
primary_frame = show_summary('best_loss')
print('EXPLORATORY PROTOCOL: best validation ACCURACY checkpoints')
exploratory_frame = show_summary('best_accuracy')

print('Reminder: 51-image held-out test set never loaded or evaluated.')


## 10. Interpretation and Stage 4 decision gate

1. **Do not select a model from a single peak epoch.** Check the primary three-seed mean and standard deviation of macro F1 using the saved best-loss checkpoints. Look at per-class recall and any worst-seed collapse. Best-accuracy checkpoints answer the secondary question of whether C08's 50/52 performance occurs across *different* random initializations.
2. A model that wins on these 52 validation images **may not** generalize better. In particular, selecting among many architectures, CLAHE settings, and epochs introduces selection bias, and initialization stability does not repair potential patient overlap or acquisition/color shortcuts.
3. **After we review Stage 3F together:** lock the winner, its preprocessing and checkpoint-selection rule, and save those decisions **before** Stage 4. Evaluate the reserved 51-image test set **once**. Do not iterate on the test results.
4. A website can demonstrate research predictions, with clear non-diagnostic disclaimers. Do not upload raw X-rays or private manifests to a public GitHub repository.

**Next:** Send me screenshots of the two per-seed and summary tables after all nine runs complete. If you run only seed 123 first, send that provisional output and then continue the remaining seeds later.
